In [ ]:
import sys
import os
import multiprocessing

# CRITICAL: Set multiprocessing start method to 'spawn' BEFORE any CUDA initialization
# This fixes the "Cannot re-initialize CUDA in forked subprocess" error in Jupyter
multiprocessing.set_start_method('spawn', force=True)

# Add paths to import from long_form_factuality
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
lff_root = os.path.join(project_root, "long_form_factuality")
for path in [project_root, lff_root]:
    if path not in sys.path:
        sys.path.insert(0, path)

from vllm_wrapper import VLLMRaterModel
# Import the original SAFE implementation
from eval.safe.rate_atomic_fact import check_atomic_fact


/root/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-11-28 23:46:48.376563247 [W:onnxruntime:Default, device_discovery.cc:164 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:89 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"


Initializing BM25 Index... (This might take a moment)


/root/venv/lib/python3.12/site-packages/huggingface_hub/file_download.py:979: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(
Fetching 41 files: 100%|██████████| 41/41 [00:00<00:00, 1086.08it/s]
Nov 28, 2025 11:46:48 PM org.apache.lucene.store.MemorySegmentIndexInputProvider <init>
INFO: Using MemorySegmentIndexInput with Java 21; to disable start with -Dorg.apache.lucene.store.MMapDirectory.enableMemorySegments=false


BM25 Index loaded successfully.


In [ ]:
# from openai import OpenAI

# atomic_fact = "Kang Min-chul was part of an assassination attempt on the South Korean president in 1983."

# client = OpenAI(base_url="http://localhost:80/v1",api_key="")

# response = client.chat.completions.create(
#     model="openai/gpt-oss-20b",
#     messages=[
#         {"role": "system", "content": "Your job is to tell if the user's statement is true or false."},
#         {"role": "user", "content": atomic_fact},
#     ],
# )
# print(response.choices[0].message.content)

False


In [ ]:

atomic_fact = "Kang Min-chul was part of an assassination attempt on the South Korean president in 1983."


llm = VLLMRaterModel()
rating = check_atomic_fact(atomic_fact, "Ra Jong-yil", llm, max_steps=2)
print(rating[0].answer)
print(rating[0].response)

FULL PROMPT:  Instructions:
1. You have been given a STATEMENT and some KNOWLEDGE points.
2. Your goal is to try to find evidence that either supports or does not support the factual accuracy of the given STATEMENT.
3. To do this, you are allowed to issue ONE semantic search query to an indexation of all Wikipedia articles that you think will allow you to find additional useful evidence.
4. Your query should aim to obtain new information that does not appear in the KNOWLEDGE. If you have previous search results, look at the previous queries you have made and try to construct a new query that is meaningfully different from the previous queries.
5. Format your final query by putting it in a markdown code block. 
6. The original Kang Min-chul was part of an assassination attempt on the South Korean president in 1983. was gathered from a text about "Ra Jong-yil" and could thus be a useful query.
7. Make sure to think through the problem step by step before issuing your query.

KNOWLEDGE:
N

In [3]:
import json

atomic_facts = []
for line in open("data_for_git/atomic_facts.jsonl"):
    atomic_facts.append(json.loads(line))
print(atomic_facts[0])

verdicts = {}
for sentance in atomic_facts[0]["result"]["all_atomic_facts"][2:3]:
    facts = sentance["atomic_facts"]
    for fact in facts[:1]:
        print(fact)
        rating = check_atomic_fact(fact, llm)
        verdicts[fact] = rating[0]
        # print(rating[0].response)

{'id': '598f763a-d239-4b4b-8f55-a04164f6d223', 'prompt': 'Tell me a bio of Katsunosuke Hori.', 'response': 'The retrieved documents do not contain any information about Katsunosuke Hori or a person with a similar name. Therefore, I cannot provide a biographical summary based on the given context.', 'results': {'num_claims': 4, 'sentences_and_atomic_facts': [['The retrieved documents do not contain any information about Katsunosuke Hori or a person with a similar name.', ['The retrieved documents do not contain any information about Katsunosuke Hori.', 'The retrieved documents do not contain any information about a person with a similar name.']], ['Therefore, I cannot provide a biographical summary based on the given context.', ['I cannot provide a biographical summary.', 'The inability to provide a biographical summary is due to the given context.']]], 'all_atomic_facts': [{'sentence': 'The retrieved documents do not contain any information about Katsunosuke Hori or a person with a sim

KeyError: 'result'

In [ ]:
for statement, verdict in verdicts.items():
    print(statement)
    print(verdict.response)
    print(verdict.answer)


He pursued a solo career in recent years.


We need to determine if the statement "He pursued a solo career in recent years." is supported by the knowledge. The knowledge includes multiple results about various individuals leaving groups and pursuing solo careers. Let's parse.

The knowledge:

Result 3 (Score: 10.33): 
- 18 February 2015, band member Will Singe announced on Facebook that he had left the group to pursue his solo career. The remaining members of The Collective disbanded that same month to also pursue solo careers. Members. Trent Bell. Trent Bell, born , is from Townsville, Queensland. He was a student at Kirwan State High School in Kirwan, Queensland and was the school captain in Year 12. Bell left Townsville in 2009 to pursue his dream for music; he auditioned for the seventh season of "Australian Idol" but did not make the top twelve. Bell also worked as a retail

So this says Will Singe left the group to pursue solo career in Feb 2015. Also mentions Trent Bell left To

In [ ]:
print(rating[0].response)
print(rating[1])



We need to determine if the statement "The earth is larger than the moon" is supported by the given knowledge. The knowledge provided includes several excerpts about angular diameters of Pluto, Charon, Moon, etc. Let's parse the knowledge:

- The first excerpt: "of Pluto; the Sun appears much smaller, only 39 to 65 arcseconds. By comparison, the Moon as viewed from Earth has an angular diameter of only 31 minutes of arc, or just over half a degree of arc. Therefore, Charon would appear to have eight times the diameter, or 64 times the area of the Moon; this is due to Charon's proximity to Pluto rather than size, as despite having just over one-third of a Lunar radius, Earth's Moon is 20 times more distant from Earth's surface as Charon is from Pluto's. This proximity further ensures that a large proportion"

This excerpt mentions the Moon's angular diameter and compares it to Charon. It also says "despite having just over one-third of a Lunar radius". That implies the Moon's radius i

In [ ]:
from bm25 import BM25Retriever

bm25 = BM25Retriever()

print(bm25.search("What is the capital of the moon?"))

56787589-0012 8.514599800109863
60260-0025 7.213799953460693
12754505-0002 7.156000137329102
34973164-0001 7.113500118255615
58515520-0029 7.067800045013428
0
US President Donald Trump. Kim and Moon also embraced before Moon returned to South Korea. Moon revealed details of the summit in a public address on 27 May. September 2018 summit. On 13 August, it was announced that a third 2018 inter-Korean summit would be held in the North Korean capital of Pyongyang on an unspecified day in September. The meeting was designed to capitalize on what was accomplished at the previous two summits. Ri Son Gwon, the head of the North Korean delegation, told reporters that a specific date for the summit was already set, but that they wanted
1
are sometimes loosely called "proper adjectives" (and so on), but not in mainstream linguistics. Which of these items are capitalized may be merely conventional. "Abrahamic", "Buddhist", "Hollywoodize", "Freudianism", and "Reagonomics" are capitalized; "quixotic